# Baseline: Dest 24h Average Delay (ArrDelay)

Baseline definition: predict `ArrDelay` using the mean `ArrDelay` observed at the *destination airport* (`Dest`) over the **last 24 hours**, where the history is based on **arrivals** (because delays are only known once a flight arrives) and is computed **up to the prediction time** (here: scheduled departure time in UTC).

In [1]:
import os
from dotenv import load_dotenv

import polars as pl

load_dotenv('/home/justus/code/ie500_data_mining_project/.env')

TARGET_COL = 'ArrDelay'
DEST_COL = 'Dest'
PRED_TIME_COL = 'CRSDepDateTime_UTC'
HIST_TIME_COL = 'ArrDateTime_UTC'
WINDOW = '24h'

TRAIN_PATH = 's3://data-mining/data/joined_1704/train_data.parquet'
TEST_PATH = 's3://data-mining/data/joined_1704/test_data.parquet'

storage = {
    'aws_access_key_id': os.environ['AWS_ACCESS_KEY_ID'],
    'aws_secret_access_key': os.environ['AWS_SECRET_ACCESS_KEY'],
    'aws_endpoint_url': os.environ['MLFLOW_S3_ENDPOINT_URL'],
}

pl.__version__

'1.39.3'

In [2]:
train = pl.scan_parquet(TRAIN_PATH, storage_options=storage)
test = pl.scan_parquet(TEST_PATH, storage_options=storage)

needed = [DEST_COL, TARGET_COL, PRED_TIME_COL, HIST_TIME_COL, 'Cancelled', 'FlightId']
train_small = train.select([c for c in needed if c in train.collect_schema().names()])
test_small = test.select([c for c in needed if c in test.collect_schema().names()])

train_small.collect_schema()

Schema([('Dest', String),
        ('ArrDelay', Float64),
        ('CRSDepDateTime_UTC', Datetime(time_unit='us', time_zone='UTC')),
        ('ArrDateTime_UTC', Datetime(time_unit='us', time_zone='UTC')),
        ('Cancelled', Float64),
        ('FlightId', Int64)])

In [3]:
# Quick sanity checks (time ranges, missingness)
ranges = pl.concat(
    [
        train_small.select(
            pl.lit('train').alias('split'),
            pl.len().alias('n'),
            pl.col(PRED_TIME_COL).min().alias('min_pred_time'),
            pl.col(PRED_TIME_COL).max().alias('max_pred_time'),
            pl.col(HIST_TIME_COL).min().alias('min_hist_time'),
            pl.col(HIST_TIME_COL).max().alias('max_hist_time'),
            pl.col(TARGET_COL).is_null().mean().alias('target_null_rate'),
        ),
        test_small.select(
            pl.lit('test').alias('split'),
            pl.len().alias('n'),
            pl.col(PRED_TIME_COL).min().alias('min_pred_time'),
            pl.col(PRED_TIME_COL).max().alias('max_pred_time'),
            pl.col(HIST_TIME_COL).min().alias('min_hist_time'),
            pl.col(HIST_TIME_COL).max().alias('max_hist_time'),
            pl.col(TARGET_COL).is_null().mean().alias('target_null_rate'),
        ),
    ],
    how='vertical',
)

ranges.collect()

split,n,min_pred_time,max_pred_time,min_hist_time,max_hist_time,target_null_rate
str,u32,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",f64
"""train""",13253650,2013-01-01 07:51:00 UTC,2019-01-01 09:59:00 UTC,2013-01-01 11:00:00 UTC,2019-01-01 20:07:00 UTC,0.014279
"""test""",5038533,2018-01-01 07:20:00 UTC,2020-01-01 09:59:00 UTC,2018-01-01 10:19:00 UTC,2020-01-02 03:48:00 UTC,0.016985


## Build History Table (Train Only)
We build a stream of *arrivals* at each destination from the training set, and compute a destination-specific rolling mean delay over the last 24 hours.

Then, for each test row, we as-of join to the most recent prior arrival at that same destination and use its rolling mean as the prediction.

In [4]:
# Global fallback (train only)
global_mean = (
    train_small
    .select(pl.col(TARGET_COL).mean())
    .collect()
    .item()
)
global_mean

4.769836549029741

In [5]:
arrivals = (
    train_small
    .select(
        pl.col(DEST_COL),
        pl.col(HIST_TIME_COL),
        pl.col(TARGET_COL),
    )
    .drop_nulls([DEST_COL, HIST_TIME_COL, TARGET_COL])
    .sort([DEST_COL, HIST_TIME_COL])
    .with_columns(
        pl.col(TARGET_COL)
        .rolling_mean_by(HIST_TIME_COL, window_size=WINDOW, closed='left')
        .over(DEST_COL)
        .alias('dest_mean_arrdelay_24h')
    )
)

arrivals.select(pl.len().alias('n_arrivals'), pl.col('dest_mean_arrdelay_24h').is_null().mean().alias('rolling_null_rate')).collect()

n_arrivals,rolling_null_rate
u32,f64
13064407,0.000004


## Predict on Test

In [6]:
pred = (
    test_small
    .select([c for c in [DEST_COL, PRED_TIME_COL, TARGET_COL, 'Cancelled', 'FlightId'] if c in test_small.collect_schema().names()])
    .sort([DEST_COL, PRED_TIME_COL])
    .join_asof(
        arrivals.select([DEST_COL, HIST_TIME_COL, 'dest_mean_arrdelay_24h']),
        left_on=PRED_TIME_COL,
        right_on=HIST_TIME_COL,
        by=DEST_COL,
        strategy='backward',
    )
    .with_columns(
        pl.col('dest_mean_arrdelay_24h').fill_null(global_mean).alias('yhat'),
        pl.col('dest_mean_arrdelay_24h').is_null().alias('used_global_fallback'),
    )
)

pred.select(
    pl.len().alias('n'),
    pl.col('used_global_fallback').mean().alias('global_fallback_rate'),
).collect()

/tmp/ipykernel_27048/1913334208.py:21: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  ).collect()


n,global_fallback_rate
u32,f64
5038533,1.9847e-7


## Evaluate (Test Rows With Non-Null ArrDelay)

In [11]:
eval_df = pred.filter(pl.col(TARGET_COL).is_not_null())

metrics = eval_df.select(
    pl.len().alias('n_eval'),
    (pl.col('yhat') - pl.col(TARGET_COL)).abs().mean().alias('MAE'),
    ((pl.col('yhat') - pl.col(TARGET_COL)) ** 2).mean().alias('MSE'),
    ((pl.col('yhat') - pl.col(TARGET_COL)) ** 2).mean().sqrt().alias('RMSE'),
)

metrics.collect()

/tmp/ipykernel_27048/2445632228.py:10: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  metrics.collect()


n_eval,MAE,MSE,RMSE
u32,f64,f64,f64
4952952,24.093315,2172.050824,46.605266


In [13]:
# Compare against a trivial constant predictor = global mean ArrDelay (train)
const_metrics = eval_df.select(
    pl.len().alias('n_eval'),
    (pl.lit(global_mean) - pl.col(TARGET_COL)).abs().mean().alias('MAE'),
    ((pl.col('yhat') - pl.col(TARGET_COL)) ** 2).mean().alias('MSE'),
    ((pl.lit(global_mean) - pl.col(TARGET_COL)) ** 2).mean().sqrt().alias('RMSE'),
)

pl.concat([
    metrics.with_columns(pl.lit('dest_24h_mean').alias('model')),
    const_metrics.with_columns(pl.lit('global_mean').alias('model')),
], how='vertical').select(['model','n_eval','MAE','MSE', 'RMSE']).collect()

/tmp/ipykernel_27048/1730369877.py:12: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  ], how='vertical').select(['model','n_eval','MAE','MSE', 'RMSE']).collect()


model,n_eval,MAE,MSE,RMSE
str,u32,f64,f64,f64
"""dest_24h_mean""",4952952,24.093315,2172.050824,46.605266
"""global_mean""",4952952,24.127231,2172.050824,46.347838


In [9]:
# Optional: where does it work best/worst? (top Dest by volume)
if DEST_COL in eval_df.collect_schema().names():
    by_dest = (
        eval_df
        .group_by(DEST_COL)
        .agg(
            pl.len().alias('n'),
            (pl.col('yhat') - pl.col(TARGET_COL)).abs().mean().alias('MAE'),
        )
        .sort('n', descending=True)
        .head(20)
    )
    by_dest.collect()

/tmp/ipykernel_27048/2363151536.py:13: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  by_dest.collect()
